# Pressure-Level Diagnostic

Check whether ERA5 surface pressure below 1000 hPa explains the sounding-validation outliers.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from src.stability.validation import SEASON_MONTHS, agreement_statistics
from src.viz import plotting_config

pairs = xr.open_zarr(project_root / "data/interim/sounding_era5_pairs.zarr", consolidated=False)
era5 = xr.open_zarr(project_root / "data/interim/era5_spb.zarr", consolidated=False)
if "sp" not in era5:
    raise RuntimeError(f"ERA5 cache has no 'sp'. Available variables: {list(era5.data_vars)}")

lat, lon = pairs.attrs["era5_latitude"], pairs.attrs["era5_longitude"]
sp_hpa = era5["sp"].sel(latitude=lat, longitude=lon, method="nearest").sel(time=pairs.time) / 100.0
paired = pairs.assign(surface_pressure_hPa=sp_hpa.astype("float32"))
valid = np.isfinite(paired.ri_sounding) & np.isfinite(paired.ri_era5) & np.isfinite(paired.surface_pressure_hPa)
paired_valid = paired.where(valid, drop=True)

sample_i = int(np.flatnonzero(np.isfinite(paired.surface_pressure_hPa.values))[0])
print(f"ERA5 cell: lat={lat}, lon={lon}")
print(f"Join sample: {pd.Timestamp(paired.time.values[sample_i])} -> {float(paired.surface_pressure_hPa.values[sample_i]):.2f} hPa")
print(f"Total paired timestamps N={paired.sizes['time']}; finite Ri validation N={paired_valid.sizes['time']}")

ERA5 cell: lat=60.0, lon=30.75
Join sample: 2014-01-01 00:00:00 -> 1017.72 hPa
Total paired timestamps N=8036; finite Ri validation N=5188


In [2]:
def pressure_fractions(ds):
    months = pd.DatetimeIndex(ds.time.values).month
    rows = [{"season": "ALL", "N": ds.sizes["time"], "fraction_sp_lt_1000": float((ds.surface_pressure_hPa < 1000).mean())}]
    for season, season_months in SEASON_MONTHS.items():
        mask = np.isin(months, season_months)
        rows.append({"season": season, "N": int(mask.sum()), "fraction_sp_lt_1000": float((ds.surface_pressure_hPa.values[mask] < 1000).mean())})
    return pd.DataFrame(rows).set_index("season")

pressure_all = pressure_fractions(paired)
pressure_valid = pressure_fractions(paired_valid)
print("Surface-pressure fractions, all paired timestamps:")
print(pressure_all.to_string(formatters={"fraction_sp_lt_1000": "{:.3%}".format}))
print("\nSurface-pressure fractions, finite Ri validation samples:")
print(pressure_valid.to_string(formatters={"fraction_sp_lt_1000": "{:.3%}".format}))

seasonal = pressure_all.drop(index="ALL")["fraction_sp_lt_1000"]
if seasonal.idxmax() != "DJF":
    raise RuntimeError(f"DJF is not the highest low-pressure season: {seasonal.to_dict()}")

low = paired.surface_pressure_hPa.values < 1000
finite_sounding = np.isfinite(paired.ri_sounding.values)
finite_era5 = np.isfinite(paired.ri_era5.values)
print(f"\nLow-pressure rows: {low.sum()} total; {(low & finite_sounding & finite_era5).sum()} finite Ri pairs; {(low & ~finite_sounding).sum()} with non-finite sounding Ri")

Surface-pressure fractions, all paired timestamps:
           N fraction_sp_lt_1000
season                          
ALL     8036             16.152%
DJF     1986             27.694%
MAM     2024             15.119%
JJA     2024              8.251%
SON     2002             13.736%

Surface-pressure fractions, finite Ri validation samples:
           N fraction_sp_lt_1000
season                          
ALL     5188              0.251%
DJF     1060              0.472%
MAM     1312              0.305%
JJA     1416              0.141%
SON     1400              0.143%

Low-pressure rows: 1298 total; 13 finite Ri pairs; 1285 with non-finite sounding Ri


In [3]:
def summarize(name, ds):
    stats = agreement_statistics(ds)
    miss_den = stats["false_negative"] + stats["true_positive"]
    return {
        "subset": name,
        "N": stats["n"],
        "Pearson r": stats["pearson_r"],
        "Spearman r": stats["spearman_r"],
        "RMSE": stats["rmse"],
        "bias": stats["bias"],
        "hit rate": stats["hit_rate"],
        "false alarm rate": stats["false_alarm_rate"],
        "miss rate": stats["false_negative"] / miss_den if miss_den else np.nan,
    }

subsets = {
    "full": paired_valid,
    "clean_sp_ge_1005": paired_valid.where(paired_valid.surface_pressure_hPa >= 1005, drop=True),
    "underground_sp_lt_1000": paired_valid.where(paired_valid.surface_pressure_hPa < 1000, drop=True),
}
stats_df = pd.DataFrame([summarize(name, ds) for name, ds in subsets.items()])
print(stats_df.to_string(index=False, formatters={col: "{:.3f}".format for col in stats_df.columns if col not in ("subset", "N")}))

                subset    N Pearson r Spearman r     RMSE   bias hit rate false alarm rate miss rate
                  full 5188     0.005      0.680 2394.361 59.033    0.777            0.124     0.223
      clean_sp_ge_1005 5167     0.005      0.682 2399.208 59.192    0.777            0.122     0.223
underground_sp_lt_1000   13     0.992      0.615  156.961 44.243    1.000            0.455     0.000


In [4]:
fig_path = project_root / "docs/figures/sounding_validation_pressure_diagnostic.pdf"
fig_path.parent.mkdir(parents=True, exist_ok=True)
bins = np.linspace(-50, 50, 81)

with plt.rc_context(plotting_config()):
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), sharex=True, sharey=True, constrained_layout=True)
    for ax, (name, ds) in zip(axes, subsets.items()):
        bias = (ds.ri_era5 - ds.ri_sounding).values
        bias = bias[np.isfinite(bias)]
        outside = np.mean(np.abs(bias) > 50) if len(bias) else np.nan
        ax.hist(np.clip(bias, -50, 50), bins=bins, color="#3b82a0", edgecolor="white", linewidth=0.25)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{name}\nN={len(bias)}, |bias|>50: {outside:.1%}")
        ax.set_xlabel("ERA5 - sounding Ri (clipped)")
    axes[0].set_ylabel("Count")
    fig.savefig(fig_path)
    plt.close(fig)
print(f"Saved {fig_path.relative_to(project_root)}")

Saved docs/figures/sounding_validation_pressure_diagnostic.pdf


In [5]:
note_path = project_root / "docs/notes/pressure_diagnostic_status.md"
note_path.parent.mkdir(parents=True, exist_ok=True)
fmt_pct = lambda x: f"{100 * x:.1f}%"
full, clean, underground = [stats_df.iloc[i] for i in range(3)]
sensible = clean["Pearson r"] > 0.4 and clean["RMSE"] < 5
season_text = ", ".join(f"{s} {fmt_pct(pressure_all.loc[s, 'fraction_sp_lt_1000'])}" for s in ["DJF", "MAM", "JJA", "SON"])
valid_season_text = ", ".join(f"{s} {fmt_pct(pressure_valid.loc[s, 'fraction_sp_lt_1000'])}" for s in ["DJF", "MAM", "JJA", "SON"])
note = (
    "# Pressure Diagnostic Status\n\n"
    f"Across all {int(pressure_all.loc['ALL', 'N'])} paired timestamps, ERA5 surface pressure is below 1000 hPa in {fmt_pct(pressure_all.loc['ALL', 'fraction_sp_lt_1000'])} of cases ({season_text}). "
    f"After applying the finite-Ri validation mask used by `agreement_statistics()`, only {int(pressure_valid.loc['ALL', 'N'] * pressure_valid.loc['ALL', 'fraction_sp_lt_1000'])}/{int(pressure_valid.loc['ALL', 'N'])} samples ({fmt_pct(pressure_valid.loc['ALL', 'fraction_sp_lt_1000'])}) remain underground ({valid_season_text}); 1,285 of 1,298 low-pressure paired timestamps have non-finite sounding Ri. "
    f"The full validation set has N={int(full.N)}, Pearson r={full['Pearson r']:.3f}, Spearman r={full['Spearman r']:.3f}, RMSE={full.RMSE:.1f}, bias={full.bias:.1f}, hit rate={fmt_pct(full['hit rate'])}, false alarm rate={fmt_pct(full['false alarm rate'])}, and miss rate={fmt_pct(full['miss rate'])}. "
    f"The clean subset with surface pressure at or above 1005 hPa has N={int(clean.N)}, Pearson r={clean['Pearson r']:.3f}, Spearman r={clean['Spearman r']:.3f}, RMSE={clean.RMSE:.1f}, bias={clean.bias:.1f}, hit rate={fmt_pct(clean['hit rate'])}, false alarm rate={fmt_pct(clean['false alarm rate'])}, and miss rate={fmt_pct(clean['miss rate'])}. "
    f"The underground subset has only N={int(underground.N)} finite samples, with Pearson r={underground['Pearson r']:.3f}, Spearman r={underground['Spearman r']:.3f}, RMSE={underground.RMSE:.1f}, bias={underground.bias:.1f}, hit rate={fmt_pct(underground['hit rate'])}, false alarm rate={fmt_pct(underground['false alarm rate'])}, and miss rate={fmt_pct(underground['miss rate'])}. "
    f"Removing underground hours does {'bring' if sensible else 'not bring'} the continuous metrics to the sensible target of Pearson r > 0.4 and RMSE < 5 Ri units, so the underground-pressure mechanism is {'supported' if sensible else 'not the dominant pathology and a second issue exists'}. "
    "Because the finite validation sample has very few underground cases even in DJF, the DJF-vs-JJA hit-rate disparity is not explained by underground 1000 hPa levels; the recommendation is to reconsider before committing to the 950/925 hPa download and adaptive-level refit.\n"
)
note_path.write_text(note)
print(f"Saved {note_path.relative_to(project_root)}")

Saved docs/notes/pressure_diagnostic_status.md
